# Reflector Node — Unit Tests
Step-by-step testing of the Reflector Node functions.
Run each cell independently to inspect intermediate outputs and logs.

**Pre-requisites before running:**
1. `OPENAI_API_KEY` set in `.env`
2. Qdrant running with the `openapi_reference` collection indexed (run `test_rag.ipynb` first)

In [1]:
# Step 1 — Imports and setup
# Run this cell first before any other cell.

from config import get_logger
from config.paths import TSPEC_DATA_TEST_FILE
from nodes.reader import reader_node
from nodes.planner import planner_node
from nodes.extractor import extractor_node
from nodes.reflector import reflector_node, _build_rag_query
from schemas.rules import ReflectedRule

logger = get_logger(__name__)
logger.info("Imports OK.")

/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-30 21:49:39 [INFO] __main__: Imports OK.


Using Device: cuda


In [2]:
# Step 2 — Run Reader → Planner → Extractor to prepare state
# The Reflector depends on raw_rules from the Extractor.
# Extractor is limited to 2 high-priority sections to keep cost low.

reader_result = reader_node({
    "main_doc_path": TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
})
logger.info(f"Reader OK — {len(reader_result['parsed_sections'])} section(s).")

planner_result = planner_node({
    **reader_result,
    "main_doc_path": TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
})
n_planned = len(planner_result["extraction_plan"]["sections_to_extract"])
logger.info(f"Planner OK — {n_planned} section(s) in plan.")

# Limit to 2 high-priority sections for cost control
full_plan = planner_result["extraction_plan"]
high_priority = [s for s in full_plan["sections_to_extract"] if s["priority"] == "high"]
test_plan = {**full_plan, "sections_to_extract": high_priority[:2]}

extractor_result = extractor_node({
    **reader_result,
    "main_doc_path":       TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
    "extraction_plan":     test_plan,
    "validation_errors":   [],
    "iteration_count":     0,
})

assert len(extractor_result["raw_rules"]) > 0, "Extractor returned no rules"
logger.info(f"Extractor OK — {len(extractor_result['raw_rules'])} raw rule(s) ready for Reflector.")

2026-03-30 21:49:39 [INFO] nodes.reader: Reader Node started.
2026-03-30 21:49:39 [DEBUG] nodes.reader: Parsed 238 relevant sections from main doc.
2026-03-30 21:49:39 [DEBUG] nodes.reader: Loaded 0 auxiliary document(s).
2026-03-30 21:49:39 [INFO] tools.document_tools: discover_specs: directory mode — found 1 .md files
2026-03-30 21:49:39 [DEBUG] tools.document_tools: discover_specs: returning 1 entries
2026-03-30 21:49:39 [DEBUG] tools.document_tools: Loaded 1 OpenAPI reference file(s) from '/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_rulesbank/src/config/../../data/references/openapi_reference'.
2026-03-30 21:49:39 [INFO] nodes.reader: Reader Node complete.
2026-03-30 21:49:39 [INFO] __main__: Reader OK — 238 section(s).
2026-03-30 21:49:39 [INFO] nodes.planner: Planner Node started.
2026-03-30 21:49:39 [DEBUG] nodes.planner: Sending 238 section(s) to Planner LLM.
2026-03-30 21:49:39 [DEBUG] open

In [3]:
# Step 3 — Test _build_rag_query()
# Verifies the RAG query is built from rule_text + openapi_object + openapi_field.

sample_rule = extractor_result["raw_rules"][0]
query = _build_rag_query(sample_rule)

assert isinstance(query, str) and len(query) > 0

logger.info(f"_build_rag_query OK")
logger.info(f"  rule_text      : {sample_rule['rule_text']}")
logger.info(f"  openapi_object : {sample_rule['openapi_mapping']['openapi_object']}")
logger.info(f"  openapi_field  : {sample_rule['openapi_mapping']['openapi_field']}")
logger.info(f"  → RAG query    : {query}")

2026-03-30 21:50:24 [INFO] __main__: _build_rag_query OK
2026-03-30 21:50:24 [INFO] __main__:   rule_text      : The IS operation "createMOI" maps to an OpenAPI path with HTTP method PUT at the resource URI "{MnSRoot}/ProvMnS/{MnSVersion}/{URI-LDN-first-part}/{className}={id}".
2026-03-30 21:50:24 [INFO] __main__:   openapi_object : paths.{MnSRoot}/ProvMnS/{MnSVersion}/{URI-LDN-first-part}/{className}={id}
2026-03-30 21:50:24 [INFO] __main__:   openapi_field  : put
2026-03-30 21:50:24 [INFO] __main__:   → RAG query    : The IS operation "createMOI" maps to an OpenAPI path with HTTP method PUT at the resource URI "{MnSRoot}/ProvMnS/{MnSVersion}/{URI-LDN-first-part}/{className}={id}". paths.{MnSRoot}/ProvMnS/{MnSVersion}/{URI-LDN-first-part}/{className}={id} put


In [4]:
# Step 4 — Test reflector_node() end-to-end
# Makes one LLM call per raw rule. Uses rules from Step 2.

reflector_state = {
    **reader_result,
    **extractor_result,
    "main_doc_path":       TSPEC_DATA_TEST_FILE,
    "auxiliary_doc_paths": [],
    "extraction_plan":     test_plan,
    "validation_errors":   [],
    "iteration_count":     1,
}

reflector_result = reflector_node(reflector_state)

assert "reflected_rules" in reflector_result
assert isinstance(reflector_result["reflected_rules"], list)
assert len(reflector_result["reflected_rules"]) == len(extractor_result["raw_rules"])

flagged = [r for r in reflector_result["reflected_rules"] if r["flagged"]]
logger.info(f"reflector_node OK")
logger.info(f"  reflected_rules : {len(reflector_result['reflected_rules'])} rule(s)")
logger.info(f"  flagged         : {len(flagged)} rule(s) for priority validation")

2026-03-30 21:50:24 [INFO] nodes.reflector: Reflector Node started.
2026-03-30 21:50:24 [DEBUG] tools.rag_tools: Retrieved 5 chunk(s) for: 'The IS operation "createMOI" maps to an OpenAPI path with HT'
2026-03-30 21:50:24 [DEBUG] nodes.reflector: Reflecting rule 1/5 [322]: The IS operation "createMOI" maps to an OpenAPI path with HT
2026-03-30 21:50:24 [DEBUG] openai._base_client: Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Helper-Method': 'chat.completions.parse', 'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-df16c108-32b9-4ebe-bef8-39172666d6d9', 'post_parser': <function Completions.parse.<locals>.parser at 0x747f016760c0>, 'content': None, 'json_data': {'messages': [{'content': 'You are an expert in 3GPP technical specifications and OpenAPI design.\n\nYour task is to review an OpenAPI rule that was automatically extracted from a 3GPP\ndocument and assess its quality using Chain-of-Thought r

In [5]:
# Step 5 — Inspect reflected_rules
# Reviews confidence scores, reasoning traces, and flags.
# All rules must conform to the ReflectedRule Pydantic schema.

reflected_rules = reflector_result["reflected_rules"]

# Validate each rule matches the ReflectedRule schema
for rule_dict in reflected_rules:
    ReflectedRule(**rule_dict)  # raises ValidationError if malformed

logger.info(f"All {len(reflected_rules)} rule(s) are valid ReflectedRule objects.")
logger.info("")

for i, rule in enumerate(reflected_rules):
    flag_label = " [FLAGGED]" if rule["flagged"] else ""
    logger.info(f"Rule {i + 1} — [{rule['section_id']}]{flag_label}")
    logger.info(f"  rule_text   : {rule['rule_text']}")
    logger.info(f"  confidence  : {rule['confidence']:.2f}")
    logger.info(f"  reasoning   : {rule['reasoning'][:200]}")
    logger.info(f"  rag_context : {len(rule['rag_context'])} chars retrieved")
    logger.info("")

2026-03-30 21:50:47 [INFO] __main__: All 5 rule(s) are valid ReflectedRule objects.
2026-03-30 21:50:47 [INFO] __main__: 
2026-03-30 21:50:47 [INFO] __main__: Rule 1 — [322]
2026-03-30 21:50:47 [INFO] __main__:   rule_text   : The IS operation "createMOI" maps to an OpenAPI path with HTTP method PUT at the resource URI "{MnSRoot}/ProvMnS/{MnSVersion}/{URI-LDN-first-part}/{className}={id}".
2026-03-30 21:50:47 [INFO] __main__:   confidence  : 0.85
2026-03-30 21:50:47 [INFO] __main__:   reasoning   : 1. The rule states that the IS operation "createMOI" maps to an OpenAPI path with HTTP method PUT at a specific resource URI pattern. This is a typical mapping in 3GPP specifications where create oper
2026-03-30 21:50:47 [INFO] __main__:   rag_context : 4424 chars retrieved
2026-03-30 21:50:47 [INFO] __main__: 
2026-03-30 21:50:47 [INFO] __main__: Rule 2 — [322]
2026-03-30 21:50:47 [INFO] __main__:   rule_text   : The IS operation "getMOIAttributes" maps to an OpenAPI path with HTTP method G